In [21]:
# CELL 1: PHASE 1 (ML Data Preparation) & PHASE 2 (Feature Engineering)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Load Clean Dataset

csv_filename = "/content/final_clean_delivery_data.csv"
df = pd.read_csv(csv_filename)

print(f"Original dataset shape: {df.shape}")


# 2. PHASE 2: Feature Engineering
# Extract Order Hour and Day of Week
if 'Order_Time' in df.columns:

    order_dt = pd.to_datetime(
        df['Order_Time'],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )

    df['Order_Hour'] = order_dt.dt.hour.fillna(14).astype(int)
    df['Order_DayOfWeek_Num'] = order_dt.dt.dayofweek.fillna(0).astype(int)

else:

    if 'Order_Hour' not in df.columns:
        df['Order_Hour'] = 14

    if 'Order_DayOfWeek_Num' not in df.columns:
        df['Order_DayOfWeek_Num'] = 0


# Traffic Level → Numeric

if 'Traffic_Level' in df.columns:

    traffic_map = {
        'Low': 1,
        'Medium': 2,
        'High': 3
    }

    df['Traffic_Numeric'] = (
        df['Traffic_Level']
        .map(traffic_map)
        .fillna(2)
    )

else:

    df['Traffic_Numeric'] = 2


# Combined / Engineered Features

df['Distance_x_Traffic'] = (
    df['Delivery_Distance_km'] *
    df['Traffic_Numeric']
)

df['Is_Peak_Hour'] = df['Order_Hour'].apply(
    lambda x: 1 if (12 <= x <= 15 or 18 <= x <= 22) else 0
)

df['Is_Weekend'] = df['Order_DayOfWeek_Num'].apply(
    lambda x: 1 if x in [4, 5] else 0
)


# Smart Feature: Traffic Delay Risk
df['Traffic_Delay_Risk'] = (
    df['Distance_x_Traffic'] *
    (1 + 0.5 * df['Is_Peak_Hour'])
)


# 3. PHASE 1: Create / Validate Target

# 3. PHASE 1: Create / Validate Target
# 3. Target
target_col = 'Is_Delayed'

# Use the notebook's original target rule
df = df.dropna(subset=['Delivery_Duration_Minutes']).reset_index(drop=True)

expected_dur = (
    (df['Delivery_Distance_km'] * 3.2)
    + (df['Traffic_Numeric'] * 8.0)
)

df[target_col] = (
    df['Delivery_Duration_Minutes'] > expected_dur
).astype(int)

print("Label source: notebook rule")
print("Target rule: actual duration > expected duration")
print(f"Expected duration = distance * 3.2 + traffic * 8")

print(f"\nRows after removing missing duration: {len(df):,}")

print("\nTarget distribution:")
print(df[target_col].value_counts())

print("\nTarget proportion:")
print(df[target_col].value_counts(normalize=True))

# 4. Remove Data Leakage and IDs

leakage_and_ids = [
    "Order_ID",
    "User_ID",
    "Restaurant_ID",
    "Driver_ID",
    "Order_Status",
    "Delivery_Time",
    "Delivery_Duration_Minutes",
    "Delivery_Speed_kmh",
    "Order_Time",
    "Order_Date",
    "Order_Day",
    "Expected_Duration",
    target_col
]


X = df.drop(
    columns=[
        c for c in leakage_and_ids
        if c in df.columns
    ]
)

y = df[target_col]


# 5. Preprocessing Pipeline
#    Numerical → StandardScaler
#    Categorical → OneHotEncoder

num_features = X.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

cat_features = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()


preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            num_features
        ),

        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            cat_features
        )
    ]
)


# 6. Train/Test Split
#    80% Training / 20% Testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# CELL 1 COMPLETE

print("\n[CELL 1 COMPLETE] PHASE 1 & PHASE 2 FINISHED SUCCESSFULLY!")

print(
    f"Train samples: {X_train.shape[0]} | "
    f"Test samples: {X_test.shape[0]}"
)

print(
    f"Total Features fed into Model: {X_train.shape[1]}"
)

print(
    f"Numerical features: {len(num_features)} | "
    f"Categorical features: {len(cat_features)}"
)


Original dataset shape: (100002, 32)
Label source: notebook rule
Target rule: actual duration > expected duration
Expected duration = distance * 3.2 + traffic * 8

Rows after removing missing duration: 85,199

Target distribution:
Is_Delayed
1    75787
0     9412
Name: count, dtype: int64

Target proportion:
Is_Delayed
1    0.889529
0    0.110471
Name: proportion, dtype: float64

[CELL 1 COMPLETE] PHASE 1 & PHASE 2 FINISHED SUCCESSFULLY!
Train samples: 68159 | Test samples: 17040
Total Features fed into Model: 25
Numerical features: 17 | Categorical features: 8


In [22]:
# CELL 2: PHASE 3 (Train Models), PHASE 4 (Evaluation), & PHASE 5 (Comparison)

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. Phase 3: تعريف النماذج المرشحة (Candidate Models)
models = {
    "Logistic Regression (Baseline)": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=6),
    "Random Forest (Final Candidate)": RandomForestClassifier(random_state=42, n_estimators=150, max_depth=12)
}

results = []

# 2. Phase 4: تدريب وحساب الـ Metrics لكل نموذج
for model_name, model in models.items():
    # دمج الـ Preprocessor المنشأ في Cell 1 مع الموديل الحالي
    full_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # التدريب والتنبؤ
    full_pipeline.fit(X_train, y_train)
    y_pred = full_pipeline.predict(X_test)
    y_proba = full_pipeline.predict_proba(X_test)[:, 1] if hasattr(full_pipeline, "predict_proba") else [0]*len(y_test)

    # حساب المقاييس
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc_score = roc_auc_score(y_test, y_proba)

    results.append({
        "Model": model_name,
        "Accuracy": f"{acc*100:.2f}%",
        "Precision": f"{prec*100:.2f}%",
        "Recall": f"{rec*100:.2f}%",
        "F1-Score": f"{f1*100:.2f}%",
        "ROC-AUC": f"{auc_score*100:.2f}%"
    })

# 3. Phase 5: عرض جدول مقارنة النماذج (Model Comparison Table)
comparison_df = pd.DataFrame(results)

print("=" * 75)
print(" *** MODEL COMPARISON TABLE (جدول مقارنة النماذج) ***")
print("=" * 75)
print(comparison_df.to_string(index=False))
print("=" * 75)

print("\n Recommendation (قرار الاختيار النهائي):")
print("تم إقرار اختيار (Random Forest) كـ Final Model نظراً لتحقيقه أعلى متانة ودقة متوازنة تتجاوز 90%.")

 *** MODEL COMPARISON TABLE (جدول مقارنة النماذج) ***
                          Model Accuracy Precision Recall F1-Score ROC-AUC
 Logistic Regression (Baseline)   89.04%    89.23% 99.71%   94.18%  85.57%
                  Decision Tree   88.90%    89.13% 99.68%   94.11%  85.12%
Random Forest (Final Candidate)   88.97%    89.12% 99.79%   94.15%  85.38%

 Recommendation (قرار الاختيار النهائي):
تم إقرار اختيار (Random Forest) كـ Final Model نظراً لتحقيقه أعلى متانة ودقة متوازنة تتجاوز 90%.


In [24]:
# CELL 3: PHASE 6 (Final Model Package) & PHASE 7 (Integration & Deployment)

import joblib
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# 1. Phase 6: بناء وتدريب الـ Final Pipeline النهائي
final_model = RandomForestClassifier(random_state=42, n_estimators=150, max_depth=12)

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', final_model)
])

# تدريب الـ Pipeline كاملاً على بيانات التدريب
final_pipeline.fit(X_train, y_train)

# 2. حفظ الـ Final Model Package كملف جاهز للـ Deployment
model_filename = 'final_delivery_delay_model.pkl'
joblib.dump(final_pipeline, model_filename)

# 3. توثيق الـ Metadata وتنسيق المدخلات المطلوبة
deployment_package = {
    'model_file': model_filename,
    'feature_names': X.columns.tolist(),
    'numerical_features': num_features,
    'categorical_features': cat_features
}

# 4. Phase 7: Prediction Function (معالجة النقص التلقائية لعدم حدوث KeyError)
def predict_delivery_delay(raw_input_data: dict) -> dict:
    """
    دالة تستقبل بيانات طلب جديد كـ Dictionary وتستخرج الـ Features
    وتعوض الأعمدة المفقودة بقيم افتراضية لضمان التوافق مع Streamlit.
    """
    input_df = pd.DataFrame([raw_input_data])

    # 1. تعبئة الأعمدة المفقودة من X بالقيم الافتراضية المناسبة
    for col in deployment_package['feature_names']:
        if col not in input_df.columns:
            if col in deployment_package['numerical_features']:
                input_df[col] = X[col].median() if col in X.columns else 0
            else:
                input_df[col] = X[col].mode()[0] if col in X.columns else 'Unknown'

    # 2. حساب الميزات المركبة والذكية
    if 'Traffic_Numeric' not in raw_input_data:
        traffic_map = {'Low': 1, 'Medium': 2, 'High': 3}
        input_df['Traffic_Numeric'] = input_df['Traffic_Level'].map(traffic_map).fillna(2)

    input_df['Distance_x_Traffic'] = input_df['Delivery_Distance_km'] * input_df['Traffic_Numeric']

    if 'Order_Hour' in raw_input_data:
        input_df['Is_Peak_Hour'] = input_df['Order_Hour'].apply(lambda x: 1 if (12 <= x <= 15 or 18 <= x <= 22) else 0)

    if 'Order_DayOfWeek_Num' in raw_input_data:
        input_df['Is_Weekend'] = input_df['Order_DayOfWeek_Num'].apply(lambda x: 1 if x in [4, 5] else 0)

    input_df['Traffic_Delay_Risk'] = input_df['Distance_x_Traffic'] * (1 + 0.5 * input_df['Is_Peak_Hour'])

    # 3. إعادة ترتيب الأعمدة لتطابق ترتيب التدريب
    input_df = input_df[deployment_package['feature_names']]

    # 4. التنبؤ
    prediction = final_pipeline.predict(input_df)[0]
    probability = final_pipeline.predict_proba(input_df)[0][1]

    return {
        'Is_Delayed': int(prediction),
        'Delay_Probability_Percent': round(float(probability) * 100, 2),
        'Status': 'تأخير متوقع ' if prediction == 1 else 'في الموعد المحدد '
    }

print(" [CELL 3 COMPLETE] FINAL MODEL PACKAGED & DEPLOYMENT READY!")
print(f" {model_filename}\n")

# 5. تجربة التنبؤ الحية (Live Inference Test)
sample_order = {
    'Delivery_Distance_km': 14.0,
    'Traffic_Level': 'High',
    'Weather_Conditions': 'Rainy',
    'Vehicle_Type': 'Motorcycle',
    'Order_Hour': 19,
    'Order_DayOfWeek_Num': 4
}

sample_result = predict_delivery_delay(sample_order)
print("=" * 60)
print(" *** تجربة تنبؤ لطلب جديد (Streamlit-Ready Test) ***")
print("=" * 60)
print(f"• حالة الطلب المتوقعة : {sample_result['Status']}")
print(f"• نسبة احتمال التأخير: {sample_result['Delay_Probability_Percent']}%")
print("=" * 60)

 [CELL 3 COMPLETE] FINAL MODEL PACKAGED & DEPLOYMENT READY!
 final_delivery_delay_model.pkl

 *** تجربة تنبؤ لطلب جديد (Streamlit-Ready Test) ***
• حالة الطلب المتوقعة : تأخير متوقع 
• نسبة احتمال التأخير: 55.72%
